# Vehicle Loan Default Prediction
Machine Learning (ML) Zoomcamp Capstone #1 Project

## Objectives

- [ ] Consider traits and insights that can be gleaned from the dataset to guide / understand the impacts on employees.
- [ ] Targeting the `Attrition` feature, build a machine learning model to predict whether a given employee is likely to churn
- [ ] Rank features that promote or reduce the likelihood of churn
- [ ] "Operationalise" the machine learning model by wrapping it in an API, itself wrapped into a Docker container to create a microservice that can be used directly from any application that supports REST integration
- [ ] Document findings, ML model design and selection, the API, and Docker container

This showcase demonstrates improved competency via:

- [x] Rich output in scripts and notebooks
- [ ] chained dataframe cleaning, eda, plots, etc => vastly reduce memory allocation / optimise for garbage collection => performance gains
- [ ] feature correlation, feature selection, and plots
- [ ] automated hyperparameter tuning
- [ ] train vs validation data ROC plots, showing tuning outcomes per model type -> now choose
- [ ] actual vs predicted plots, showing tuning outcomes per model type -> now choose
- [ ] kind (kubernetes local) deployment

## Imports

In [5]:
%%time

from IPython.display import display
from rich.jupyter import print
%load_ext rich

import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score, roc_curve, root_mean_squared_error # , f1_score, mutual_info_score, auc
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text #, plot_tree, export_graphviz
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from itertools import combinations
from typing import Iterable, TypeVar, Optional

skPredict = TypeVar("skPredict", LogisticRegression, DecisionTreeClassifier, RandomForestClassifier)
skDecide = TypeVar("skDecide", skPredict, LinearRegression, DecisionTreeRegressor, RandomForestRegressor)
skFit = TypeVar("skFit", skPredict, skDecide)

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from tqdm.auto import tqdm

import xgboost as xgb

CPU times: total: 1.31 s
Wall time: 2.08 s


### Display versions

In [15]:
%%time 

from platform import python_version
import arff
import sklearn as sk
import matplotlib as mpl
import tqdm as tq
import subprocess as cmd

versions = {
    "python": python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "liac-arff": arff.__version__,
    "sklearn": sk.__version__,
    "matplotlib": mpl.__version__,
    "seaborn": sns.__version__,
    "tqdm": tq.__version__,
}

for line in cmd.check_output(["jupyter", "--version"]).decode().split('\n'):
    comps = list( map(str.strip, line.split(': ')) )
    if len(comps) == 2:
        versions[comps[0]] = comps[1]
    
print( dict( sorted(versions.items() )))

del python_version, arff, sk, mpl, tq, cmd, line, comps, versions

{
    'IPython': '8.30.0',
    'ipykernel': '6.29.5',
    'ipywidgets': '8.1.5',
    'jupyter_client': '8.6.3',
    'jupyter_core': '5.7.2',
    'jupyter_server': 'not installed',
    'jupyterlab': 'not installed',
    'liac-arff': '2.4.0',
    'matplotlib': '3.10.0',
    'nbclient': 'not installed',
    'nbconvert': 'not installed',
    'nbformat': 'not installed',
    'notebook': 'not installed',
    'numpy': '1.26.4',
    'pandas': '2.2.3',
    'python': '3.11.6',
    'qtconsole': 'not installed',
    'seaborn': '0.13.2',
    'sklearn': '1.4.0',
    'tqdm': '4.67.1',
    'traitlets': '5.14.3'
}

CPU times: total: 31.2 ms
Wall time: 1.48 s


In [3]:
# %%time

# from rich.progress import track
# import time

# for i in track(range(100), description="Processing..."):
#     time.sleep(0.02)

## Utilities

In [60]:
strip_whitespace = lambda val: val.strip() if type(val) is str else val

In [63]:
# https://stackoverflow.com/questions/1175208/elegant-python-function-to-convert-camelcase-to-snake-case
import re

def to_snake_case(name) -> str:
    name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub('([a-z0-9])([A-Z])', r'\1_\2', name)
    return name.replace(' ','_').replace('__','_').replace('.', '_').replace('(', '').replace(')', '').lower()

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.map(to_snake_case)
    return df

# print(to_snake_case('camel2_camel2_case'))   # camel2_camel2_case
# print(to_snake_case('camel2C Camel2_Case'))  # camel2_camel2_case
# print(to_snake_case('getHTTPResponseCode'))  # get_http_response_code
# print(to_snake_case('HTTPResponseCodeXYZ'))  # http_response_code_xyz

In [44]:
def scalar_features(df: pd.DataFrame, exclude: Iterable[str] = [], ascending: Optional[bool] = None) -> Iterable[str]:
    cols = df.select_dtypes(exclude=[object, 'category']).columns
    cols = cols if ascending == None else cols.sort_values(ascending=ascending)
    return [col for col in cols if col not in exclude]

def categorical_features(df: pd.DataFrame, exclude: Iterable[str] = [], ascending: Optional[bool] = None) -> Iterable[str]:
    cols = df.select_dtypes(include=[object, 'category']).columns
    cols = cols if ascending == None else cols.sort_values(ascending=ascending)
    return [col for col in cols if col not in exclude]

In [45]:
def get_lookup_value(id: int, df_lookup: pd.DataFrame, col:str = 'level', ifNull:str = 'Not Answered') -> str:
    return f'{ int(id) } { df_lookup.loc[id][col] }' if id in df_lookup.index else ifNull

In [46]:
def validation_testing_training_full_split(dataframe: pd.DataFrame, seed: int = 42, validation: float = 0.2, testing: float = 0.2) \
    -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    assert 0 < validation and 0 < testing and 1 > (validation + testing)

    validation_of_full = validation / (1 - testing)
    if validation_of_full == 0:
        validation_of_full = None
        
    df_full,     df_testing    = train_test_split(dataframe, test_size=testing,            random_state=seed, shuffle=True)
    df_training, df_validation = train_test_split(df_full,   test_size=validation_of_full, random_state=seed, shuffle=True)
    
    df_validation = df_validation.reset_index(drop=True)
    df_testing = df_testing.reset_index(drop=True)
    df_training = df_training.reset_index(drop=True)
    df_full = df_full.reset_index(drop=True)
    
    return df_validation, df_testing, df_training, df_full

In [47]:
def y_split(dataframe: pd.DataFrame, yColumn: str, drop: Iterable[str] = []) -> tuple[pd.DataFrame, pd.Series]:
    columns = set(dataframe.columns)   
    assert columns.issuperset([yColumn]), f'{yColumn} not found in dataframe'
    assert columns.issuperset(drop), f'At least one of {drop} not found in dataframe'
    
    df = dataframe.copy()
    y = df[yColumn]
    for col in [*drop, yColumn]:
        del df[col]
        
    return df, y

In [48]:
def regularize(X, r=0.000000001):
    return X + np.eye(X.shape[0]) * r

In [49]:
# def regularize(X, r: float = 0.00000001):
#     return X if r == 1 else X + np.eye(X.shape[0]) * r
#     
# X = [
#     [1, 2, 2],
#     [2, 1, 1], # r
#     [2, 1, 1], # r
#     [0, 9, 9],
#     [1, 0, 0],
# ] #     c  c
# X = np.array(X)
# XTX = X.T.dot(X)
# print(XTX) # duplicate columns is an issue for linear / logistic regression
# np.linalg.inv(XTX)
# 
# regularize(X, r=0.01)

In [50]:
def sigmoid(score):
    return 1 / (1 + np.exp(-score))

# z = np.linspace(-5, 5, 51)
# plt.plot(z, sigmoid(z))

In [51]:
def display_predictive_features_for_target(df: pd.DataFrame, target: str, categorical: Iterable[str] = []):
    global_target = df[target].mean()
    for c in categorical:
        df_group = df.groupby(c)[target].agg('mean','count')
        df_group['diff'] = df_group.mean() - global_target
        df_group['risk'] = df_group.mean() / global_target
        display(df_group)

In [52]:
def one_hot_encode(df: pd.DataFrame, dv: DictVectorizer = DictVectorizer(sparse=False), drop: Iterable[str] = [], fit: bool = False):
    assert set(df.columns).issuperset(drop), f'At least one of {drop} is not found in the DataFrame `df`'
    
    df_encode = df.copy()
    for feature in drop:
        del df_encode[feature]
    
    data = df_encode.to_dict(orient='records')
    X = dv.fit_transform(data) if fit else dv.transform(data)
        
    assert len(dv.feature_names_) == X.shape[1]
    return X, dv

In [53]:
def fit(model: skFit, df: pd.DataFrame, y: pd.Series, dv: DictVectorizer = DictVectorizer(sparse=False), drop: Iterable[str] = []) -> tuple[skFit, DictVectorizer]:
    assert df.shape[0] == y.shape[0], '`df` and `y` mismatch'
    
    X, dv = one_hot_encode(df, dv, drop, fit=True)
    model.fit(X, y)
    
    return model, dv

In [54]:
def decide(model: skDecide, dv: DictVectorizer, df: pd.DataFrame, drop: Iterable[str] = []):
    X, _ = one_hot_encode(df, dv, drop)
    y_pred = model.predict(X)
    
    return y_pred

In [55]:
def predict(model: skPredict, dv: DictVectorizer, df: pd.DataFrame, drop: Iterable[str] = []):
    X, _ = one_hot_encode(df, dv, drop)
    y_pred = model.predict_proba(X)[:, 1]
    
    return y_pred

In [56]:
def random_predictions(y: pd.Series, seed: int = 42):
    np.random.seed(seed)
    if y.dtype == 'bool':
        return np.random.uniform(0, 1, size=len(y))
    else:
        return np.random.uniform(0, 1, size=len(y))

In [57]:
def model_metrics(y: pd.Series, y_pred: pd.Series, threshold: float = 0.5):
    assert len(y) == len(y_pred), "`y` and `y_pred` mismatch"
    assert 0 <= threshold and threshold <= 1, "invalid threshold"
    
    actual_positive = (y == 1)
    actual_negative = (y == 0)
    predicted_positive = (y_pred >= threshold)
    predicted_negative = (y_pred < threshold)
    
    true_positive = (predicted_positive & actual_positive).sum()
    false_positive = (predicted_positive & actual_negative).sum()
    false_negative = (predicted_negative & actual_positive).sum()
    true_negative = (predicted_negative & actual_negative).sum()
    
    # Accuracy = ratio of correct predictions -- false and positive 
    # accuracy = proportional_matrix[0,0] + proportional_matrix[1,1]
    accuracy = (true_positive + true_negative) / (true_positive + false_positive + false_negative + true_negative)
    
    # Precision = ratio of correct to positive predictions; higher is better
    precision = true_positive / (true_positive + false_positive)
    # Recall = ratio of correctly predicted positive; higher is better
    recall = true_positive / (true_positive + false_negative)
    f1 = 2 * (precision * recall) / (precision + recall)
    
    # TPR = ratio of true positives in all positives; higher is better
    # true_positive_rate = recall
    true_positive_rate  = true_positive / ( false_negative + true_positive)
    # FPR = ratio of false positives in all negatives; lower is better
    false_positive_rate = false_positive / (true_negative + false_positive)
    
    matrix = np.array([
        #   g(Xi) < t    |   g(Xi) >= t
        [ true_negative  , false_positive ], # y == 0
        [ false_negative , true_positive  ]  # y == 1
    ])
    
    proportional_matrix = (matrix / matrix.sum()).round(6)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': true_positive,
        'fp': false_positive,
        'fn': false_negative,
        'tn': true_negative,
        'tpr': true_positive_rate,
        'fpr': false_positive_rate,
        'confusion': matrix,
        'proportional_confusion': proportional_matrix
    }

In [58]:
def model_metrics_for_thresholds(y: pd.Series, y_pred: pd.Series, thresholds: Iterable[float]) -> pd.DataFrame:
    results = []
    for threshold in thresholds:
        metrics = model_metrics(y, y_pred, threshold)
        metrics['threshold'] = threshold
        results.append(metrics)
        
    df_metrics = pd.DataFrame(results)
    df_metrics.set_index('threshold', inplace=True)
    
    return df_metrics

In [59]:
def all_combinations_of(arr):
    return (combo for choose in range(1 + len(arr)) for combo in map(list, combinations(arr, choose)))

## Data Preparation

### Performance Ratings / `dfPerformance`:

<!---  https://www.markdownguide.org/extended-syntax/ --->

PerformanceID
:   Unique identifier for each performance review.

EmployeeID
:   Unique identifier for the employee being reviewed.

ReviewDate
:   The date of the performance review.

EnvironmentSatisfaction
:   Rating of the employee's satisfaction with their work environment.

JobSatisfaction
:   Rating of the employee's satisfaction with their job.

RelationshipSatisfaction
:   Rating of the employee's satisfaction with workplace relationships.

TrainingOpportunitiesWithinYear
:   Number of training opportunities available to the employee within the year.

TrainingOpportunitiesTaken
:   Number of training opportunities the employee has taken.

WorkLifeBalance
:   Rating of the employee's work-life balance.

SelfRating
:   The employee's self-assessment rating.

ManagerRating
:   The manager's rating of the employee's performance.


### Employees / `dfEmployee`:

EmployeeID
:   Unique identifier for each employee.

FirstName
:   The first name of the employee.

LastName
:   The last name of the employee.

Gender
:   The gender of the employee.

Age
:   The age of the employee.

BusinessTravel
:   The frequency of business travel for the employee.

Department
:   The department in which the employee works.

DistanceFromHome (KM)
:   The distance between the employee's home and workplace in kilometers.

State
:   The state in which the employee resides.

Ethnicity
:   The ethnicity of the employee.

MaritalStatus
:   The marital status of the employee.

Salary
:   The annual salary of the employee.

StockOptionLevel
:   The level of stock options granted to the employee.

OverTime
:   Whether the employee works overtime (Yes/No).

HireDate
:   The date the employee was hired.

Attrition
:   Whether the employee has left the company (Yes/No).

YearsAtCompany
:   The number of years the employee has been with the company.

YearsInMostRecentRole
:   The number of years the employee has been in their most recent role.

YearsSinceLastPromotion
:   The number of years since the employee's last promotion.

YearsWithCurrManager
:   The number of years the employee has worked with their current manager.

### Load Data

#### Load raw .arff data into a DataFrame

In [64]:
%%time
# from scipy.io import arff   # FAILED as "NotImplementedError: String attributes not supported yet"
import arff

raw = arff.load( open('./data/LT-Vehicle-Loan-Default-Prediction.arff', 'r') )

print(raw.keys())
print(raw['attributes'])

columns = [to_snake_case(name) for name, type in raw['attributes']]
print(f"{columns = }")

df_raw = pd.DataFrame(np.array(raw['data']), columns=columns)

del arff, raw, columns

dict_keys(['description', 'relation', 'attributes', 'data'])

[
    ('uniqueid', 'INTEGER'),
    ('disbursed_amount', 'INTEGER'),
    ('asset_cost', 'INTEGER'),
    ('ltv', 'REAL'),
    ('branch_id', 'INTEGER'),
    ('supplier_id', 'INTEGER'),
    ('manufacturer_id', 'INTEGER'),
    ('current_pincode_id', 'INTEGER'),
    ('date.of.birth', 'STRING'),
    ('employment.type', 'STRING'),
    ('disbursaldate', 'STRING'),
    ('state_id', 'INTEGER'),
    ('employee_code_id', 'INTEGER'),
    ('mobileno_avl_flag', 'INTEGER'),
    ('aadhar_flag', 'INTEGER'),
    ('pan_flag', 'INTEGER'),
    ('voterid_flag', 'INTEGER'),
    ('driving_flag', 'INTEGER'),
    ('passport_flag', 'INTEGER'),
    ('perform_cns.score', 'INTEGER'),
    ('perform_cns.score.description', 'STRING'),
    ('pri.no.of.accts', 'INTEGER'),
    ('pri.active.accts', 'INTEGER'),
    ('pri.overdue.accts', 'INTEGER'),
    ('pri.current.balance', 'INTEGER'),
    ('pri.sanctioned.amount', 'INTEGER'),
    ('pri.disbursed.amount', 'INTEGER'),
    ('sec.no.of.accts', 'INTEGER'),
    ('sec.active.accts', 'INTEGER'),
    ('sec.overdue.accts', 'INTEGER'),
    ('sec.current.balance', 'INTEGER'),
    ('sec.sanctioned.amount', 'INTEGER'),
    ('sec.disbursed.amount', 'INTEGER'),
    ('primary.instal.amt', 'INTEGER'),
    ('sec.instal.amt', 'INTEGER'),
    ('new.accts.in.last.six.months', 'INTEGER'),
    ('delinquent.accts.in.last.six.months', 'INTEGER'),
    ('average.acct.age', 'STRING'),
    ('credit.history.length', 'STRING'),
    ('no.of_inquiries', 'INTEGER'),
    ('loan_default', ['0', '1'])
]

columns = ['uniqueid', 'disbursed_amount', 'asset_cost', 'ltv', 'branch_id', 'supplier_id', 'manufacturer_id', 
'current_pincode_id', 'date_of_birth', 'employment_type', 'disbursaldate', 'state_id', 'employee_code_id', 
'mobileno_avl_flag', 'aadhar_flag', 'pan_flag', 'voterid_flag', 'driving_flag', 'passport_flag', 
'perform_cns_score', 'perform_cns_score_description', 'pri_no_of_accts', 'pri_active_accts', 'pri_overdue_accts', 
'pri_current_balance', 'pri_sanctioned_amount', 'pri_disbursed_amount', 'sec_no_of_accts', 'sec_active_accts', 
'sec_overdue_accts', 'sec_current_balance', 'sec_sanctioned_amount', 'sec_disbursed_amount', 'primary_instal_amt', 
'sec_instal_amt', 'new_accts_in_last_six_months', 'delinquent_accts_in_last_six_months', 'average_acct_age', 
'credit_history_length', 'no_of_inquiries', 'loan_default']

CPU times: total: 15 s
Wall time: 17.9 s


#### Clean raw data

In [65]:
df_raw

,uniqueid,disbursed_amount,asset_cost,ltv,branch_id,supplier_id,manufacturer_id,current_pincode_id,date_of_birth,employment_type,...,sec_sanctioned_amount,sec_disbursed_amount,primary_instal_amt,sec_instal_amt,new_accts_in_last_six_months,delinquent_accts_in_last_six_months,average_acct_age,credit_history_length,no_of_inquiries,loan_default
0,420825,50578,58400,89.55,67,22807,45,1441,01-01-84,Salaried,...,0,0,0,0,0,0,0yrs 0mon,0yrs 0mon,0,0
1,537409,47145,65550,73.23,67,22807,45,1502,31-07-85,Self employed,...,0,0,1991,0,0,1,1yrs 11mon,1yrs 11mon,0,1
2,417566,53278,61360,89.63,67,22807,45,1497,24-08-85,Self employed,...,0,0,0,0,0,0,0yrs 0mon,0yrs 0mon,0,0
3,624493,57513,66113,88.48,67,22807,45,1501,30-12-93,Self employed,...,0,0,31,0,0,0,0yrs 8mon,1yrs 3mon,1,1
4,539055,52378,60300,88.39,67,22807,45,1495,09-12-77,Self employed,...,0,0,0,0,0,0,0yrs 0mon,0yrs 0mon,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233149,626432,63213,105405,60.72,34,20700,48,1050,01-08-88,Salaried,...,0,0,4084,0,0,0,1yrs 9mon,3yrs 3mon,0,0
233150,606141,73651,100600,74.95,34,23775,51,990,05-12-88,Self employed,...,0,0,1565,0,0,0,0yrs 6mon,0yrs 6mon,0,0
233151,613658,33484,71212,48.45,77,22186,86,2299,01-06-76,Salaried,...,0,0,0,0,0,0,0yrs 0mon,0yrs 0mon,0,0
233152,548084,34259,73286,49.1,77,22186,86,2299,26-03-94,Salaried,...,0,0,0,0,0,0,0yrs 0mon,0yrs 0mon,0,0


In [ ]:
# Load look-up values

[dfEducation, dfRating, dfSatisfied] = map(standardize_columns, [
    pd.read_csv('./data/EducationLevel.csv', names=['id', 'level'], index_col='id', skiprows=1).map(strip_whitespace), 
    pd.read_csv('./data/RatingLevel.csv', names=['id', 'level'], index_col='id', skiprows=1).map(strip_whitespace),
    pd.read_csv('./data/SatisfiedLevel.csv', names=['id', 'level'], index_col='id', skiprows=1).map(strip_whitespace)
    ])


display(dfEducation)
display(dfRating)
display(dfSatisfied)


# Load source data

[dfEmployee, dfPerformance] = map(standardize_columns, [
    pd.read_csv('./data/Employee.csv').map(strip_whitespace), 
    pd.read_csv('./data/PerformanceRating.csv').map(strip_whitespace)
    ])

display( dfPerformance.describe(include='all').T )
display( dfEmployee.describe(include='all').T )

### Inspect datasets for preparation tasks & ideas

Glancing at the data, the following needs attention:

1. Performance data:

- [x] what to do if most-recent scores have empty values?  n/a
- [ ] `training`, when offered appears to be readily taken.  Should drive inverse correlation to `attrition`; check
- [x] `self_rating` and `manager_rating` scores never use full 1..5 range.  Would consider standardization, but int values exclude usefulness

2. Join performance to employee data frame using `employee_id`:

- [x] performance data for non-existing employees? drop records
- [x] retain only latest performance data per employee (as prior events may have triggered attrition, but employee left in last year only)
- [x] drop perf data frame

3. Employee data:

- [x] what to do if empty values?  n/a
- [x] what to do if employee has no performance data? consider `attrition`, drop?
- [x] categorical features (`overtime` & `attrition`): convert to binary
- [ ] demographic features (`first_name`, `last_name`, and perhaps `age`, `ethnicity` and `gender` if not significant to models) should be dropped
- [ ] `years_since_promotion` & `years_at_company` -> long, whereas `years_in_role` & `years_with_manager` -> short; "stickiness" should correlate to `attrition`; check
- [x] once merged, `employee_id` should replace the index -> excluded from models
- [x] once merged, compare and perhaps generate features from the `hire_date`, `review_date`, and `years_*` features
- [ ] consider correlation of `business_travel` & `job_role` to `attrition`
- [ ] only 3 `states` and 3 `departments`: can't generalise to USA. Underscores specific to this company!

4. Look-up values:

- [x] look-up values should replace references (for future one-hot encoding, else references will incorrectly be interpreted as scalars)
- [x] drop look-up data frames
- [x] `review_date` and `hire_date` str -> Date convert, and expand to 'yyyy-MM' for trend analysis

### Performance

In [ ]:
dfPerformance.isnull().sum()

In [ ]:
dfPerformance.employee_id.isin(dfEmployee.employee_id).unique()

Observations:

1.  No empty values
1.  Every `employee_id` is found in the `dfEmployee` frame, so no data to drop

### Employee

In [ ]:
dfEmployee.isnull().sum()

In [ ]:
dfEmployee[dfEmployee.employee_id.duplicated()]

In [ ]:
dfEmployee[dfEmployee.employee_id.isin(dfPerformance.employee_id) == False].attrition.value_counts()

Observations:

1. No empty values
1. No duplicate `employee_id`s
1. There are 190 employees that have no performance data. 
1. But as none have left, they can be assumed to be satisfied.
1. No data interventions (eg dropping records) necessary.

In [ ]:
dfEmployee.stock_option_level.value_counts()

### Join most-recent Performance data per `employee_id` to Employee data

In [27]:
dfPerformance.review_date = pd.to_datetime(dfPerformance.review_date)  # USA dates format is default
dfPerformance['review_date_ym'] = dfPerformance.review_date.dt.to_period('M')
dfPerformance['review_date_yq'] = dfPerformance.review_date.dt.to_period('Q')


In [ ]:
dup_perf_employee_id = dfPerformance[dfPerformance.employee_id.duplicated() == True].tail(1).employee_id.values
display(f'BEFORE: Duplicate performance records for {dup_perf_employee_id[0]}:', dfPerformance[dfPerformance.employee_id.isin(dup_perf_employee_id)])

In [ ]:
dfPerformance_recent = dfPerformance \
    .sort_values(by=['employee_id', 'review_date'], ascending=[True, False]) \
        .drop_duplicates(subset='employee_id', keep='first')

display(f'AFTER: Duplicate performance records for {dup_perf_employee_id[0]}:', dfPerformance_recent[dfPerformance_recent.employee_id.isin(dup_perf_employee_id)])

In [ ]:
display(f'BEFORE merge: {dfEmployee.shape = }')

dfEmployee = pd.merge(dfEmployee, dfPerformance_recent, how='left', on='employee_id') \
    .set_index('employee_id')

display(f'AFTER merge: {dfEmployee.shape = }.  Note: index set to `employee_id`')
dfEmployee.loc[dup_perf_employee_id].T

Observations:

1.  Merge and re-indexing successful
1.  **ISSUE WITH DATA**: 10yrs between higher and most-recent performance review, yet `dfEmployee.years_at_company = 1` doesn't reflect:
  - possible issue in HR system, or break in data maintenance
  - [x] generate further `years_*` features -> model / feature importance will show significance
  - if generated features found worthwhile -> correct source data / establish data pipeline / responsibility for caller / API

### Merge look-up values

In [ ]:
dfEmployee['education_level'] = dfEmployee.education.apply(lambda id: get_lookup_value(id, dfEducation))

dfEmployee['environment_satisfaction_level'] = dfEmployee.environment_satisfaction.apply(lambda id: get_lookup_value(id, dfSatisfied))
dfEmployee['job_satisfaction_level'] = dfEmployee.job_satisfaction.apply(lambda id: get_lookup_value(id, dfSatisfied))
dfEmployee['relationship_satisfaction_level'] = dfEmployee.relationship_satisfaction.apply(lambda id: get_lookup_value(id, dfSatisfied))
dfEmployee['work_life_balance_level'] = dfEmployee.work_life_balance.apply(lambda id: get_lookup_value(id, dfSatisfied))

dfEmployee['self_rating_level'] = dfEmployee.self_rating.apply(lambda id: get_lookup_value(id, dfRating))
dfEmployee['manager_rating_level'] = dfEmployee.manager_rating.apply(lambda id: get_lookup_value(id, dfRating))

dfEmployee.info()

In [ ]:
dfEmployee.isnull().sum()

## Feature Engineering

In [ ]:
dfEmployee.head().T

### Generate date features

In [34]:
dfEmployee.hire_date = pd.to_datetime(dfEmployee.hire_date)
dfEmployee['hire_date_ym'] = dfEmployee.hire_date.dt.to_period('M') # f'{ dfEmployee[col].dt.year }-{ dfEmployee[col].dt.month }'
dfEmployee['hire_date_yq'] = dfEmployee.hire_date.dt.to_period('Q')

In [ ]:
dfEmployee['years_since_hire'] = \
    (dfEmployee.review_date.dt.year - dfEmployee.hire_date.dt.year) \
        .fillna(0).apply(lambda y: int(y) if 0 <= y else 0)
        
dfEmployee.years_since_hire.describe()

### Refine feature data types (-> performance)

#### Categorical features

In [ ]:
for col in ['over_time', 'attrition']:
    dfEmployee[col] = dfEmployee[col].apply(lambda val: 0 if 'No' in val else 1)


for col in ['gender', 'business_travel', 'department', 'state', 'ethnicity', 'education_field', 'job_role', 'marital_status', 'stock_option_level', 'education_level',
            'environment_satisfaction_level', 'job_satisfaction_level', 'relationship_satisfaction_level', 'work_life_balance_level', 'self_rating_level', 'manager_rating_level']:
    dfEmployee[col] = dfEmployee[col].astype('category')
    
dfEmployee.info()

#### Scalar features

In [ ]:
display("BEFORE: ", dfEmployee[scalar_features(dfEmployee)].info())

dfEmployee.salary = dfEmployee.salary.astype('UInt32')

for col in dfEmployee.select_dtypes(include=[int, float]).columns:
    dfEmployee[col] = dfEmployee[col].round().astype('UInt8')

display("AFTER: ", dfEmployee[scalar_features(dfEmployee)].info())

In [ ]:
scalar_features(dfEmployee)

In [ ]:
categorical_features(dfEmployee)

## Exploratory Data Analysis (EDA)

In [ ]:
# https://seaborn.pydata.org/generated/seaborn.color_palette.html
# https://seaborn.pydata.org/tutorial/color_palettes.html
# https://www.practicalpythonfordatascience.com/ap_seaborn_palette

# sns.set_style("darkgrid")
sns.set_style("whitegrid")

# palette = sns.color_palette("flare", 8)
# palette = sns.mpl_palette("Set2")
# palette = sns.color_palette("Spectral", as_cmap=True)
# palette = sns.mpl_palette("viridis")
palette = sns.color_palette('YlGnBu_r', 8)
sns.set_palette(palette=palette)
palette

### Display `dfEmployee`

In [ ]:
dfEmployee.describe(include='all').T

### Univariate Analysis

#### Explore scalar features

In [ ]:
exclude = ['over_time', 'attrition', 'hire_date', 'hire_date_ym', 'hire_date_yq', 'review_date', 'review_date_ym', 'review_date_yq',
           # below are handled via categorical analysis
           'education', 'environment_satisfaction', 'job_satisfaction', 'manager_rating', 'relationship_satisfaction', 'self_rating', 'work_life_balance']

plt.figure(figsize=(16, 16))
for i, feat in enumerate(scalar_features(dfEmployee, exclude=exclude, ascending=True), 1):
    plt.subplot(5, 4, i)
    bins = 3 if feat in ['training_opportunities_within_year', 'training_opportunities_taken'] else 10
    sns.histplot(y=dfEmployee[feat], bins=bins, kde=True)
    
plt.tight_layout()

In [ ]:
top_decile_salary = dfEmployee.salary.describe(percentiles=[.9])['90%']
bottom_decile_salary = dfEmployee.salary.describe(percentiles=[.1])['10%']

print(f'{top_decile_salary = :>8.0f}\n{bottom_decile_salary = :8>.0f}')

top_decile_salary = dfEmployee.salary[dfEmployee.salary >= top_decile_salary].mean() #.describe()
bottom_decile_salary = dfEmployee.salary[dfEmployee.salary <= bottom_decile_salary].mean() #.describe()

print(f'\nSalary Inequality Quotient = {top_decile_salary / bottom_decile_salary:.1f}')

In [ ]:
plt.figure(figsize=(16, 16))
for i, feat in enumerate(['training_opportunities_within_year', 'training_opportunities_taken'], 1):
    _ = dfEmployee[feat].fillna(0).value_counts().sort_index(ascending=False)
    plt.subplot(3, 4, i)
    plt.pie(_, autopct='%1.0f%%', labels=_.index, explode=(0, 0, 0, 0.1))
    plt.title(_.index.name)

In [ ]:
print(f'Training take-up = { dfEmployee.training_opportunities_taken.sum() / dfEmployee.training_opportunities_within_year.sum() * 100 :.1f}%')

In [ ]:
_ = pd.merge(left=dfEmployee.hire_date_ym.value_counts(), right=dfPerformance.review_date_ym.value_counts(), how='outer', left_index=True, right_index=True)
_ = pd.merge(left=_,                                      right=dfPerformance_recent.review_date_ym.value_counts(), how='outer', left_index=True, right_index=True)
_.fillna(0, inplace=True)
_.reset_index(drop=False, inplace=True)
_.columns = ['month', 'hires', 'reviews', 'most_recent_review']
_.month = _.month.dt.strftime('%Y-%m')
_.set_index('month', inplace=True)

# display(_)

plt.figure(figsize=(20,5))
for label in sns.lineplot(data=_).get_xticklabels():
    label.set_visible(True) if label.get_text()[-2:] in ['01', '04', '07', '10'] else label.set_visible(False)
plt.xticks(rotation=65, ha='right')
plt.title('Hires and Reviews per month')

Observations:

age
:   Nothing remarkable, # IDEA but would be interesting to see contrasted against `state`, `education`, and `ethnicity`

distance_from_home_km
:   Unexpectedly uniform distribution of distances, so perhaps a remote-first organisation?

salary
:   Elevated inequality -- top decile is 14x the lowest salary decile -- but not unusual.  Perhaps a policy consideration, depending on the ethos of the organisation

training_opportunities_taken
:   When offered, training opportunities are often, but not readily, taken (48% take-up)
:   But there's also a large contingent that are not offered training, or when offered don't pursue training 
:   So there's perhaps a disincentive / counter-culture to take-up of offered training

years_at_company
:   The 10yrs decile is unexpectedly the largest, indicating founding staff are "lifers"
:   Noticed in `Field Engineering` above, when compared to `dfPerformance.review_date`, values here don't correspond. This is illustrated when comparing `years_at_company` to `years_since_hire` plots

hire_date_ym
:   Except for an initial flurry, hiring is consistent
:   As attrition isn't dated, this can't be assessed / plotted from the source data.  # IDEA Enrich source data with `attrition date = min( Resignation date, Exit date )`

review_date_ym
:   Reviews appear to be consistent, seasonal (Q1-Q2 each year), and increasing each year as hires increase headcount
:   There appears to be a progressive "late submission of reviews" trend, which may be indicative of complacency, routine, or perhaps a disincentive / reluctance to review
:   As attrition isn't dated, only the most-recent review is used for modelling below.

#### Explore categorical features

In [ ]:
plt.figure(figsize=(20, 20))
for i, feat in enumerate(categorical_features(dfEmployee, exclude=['first_name', 'last_name', 'performance_id'], ascending=True), 1):
    plt.subplot(5, 4, i)
    # sns.countplot(y=dfEmployee[feat])
    _ = dfEmployee[feat].value_counts()
    plt.pie(_, autopct='%1.0f%%', labels=_.index)
    plt.title(_.index.name)

plt.tight_layout()

Observations:

business_travel
:   90% of staff travel, which indicates this organisation may provide professional or field services

department
:   Only 3 departments are drawn from the data. There would typically be other functions in an organisation, which implies the data may be incomplete as a result of policy. This needs investigation, where staff not in the 3 known departments cannot therefore be reliably predicted using the ML models below

education_level
:   70% of staff have tertiary / university education

### Bivariate Analysis

In [ ]:
barh_specs = [
    ('job_role',   'salary', 'United States Dollar (USD)', 'Average Salary by Job Role'),
    ('department', 'salary', 'United States Dollar (USD)', 'Average Salary by Department'),
]

plt.figure(figsize=(20, 20))
for i, (group, scalar, label, title) in enumerate(barh_specs, 1):
    plt.subplot(5, 4, i)
    _ = dfEmployee.groupby(group, observed=True)[scalar].mean().reset_index().sort_values(by=group, ascending=False)
    
    plt.barh(data=_, y=_[group], width=_[scalar])
    plt.xlabel(label)
    plt.title(title)
    
plt.tight_layout()

**NOTE:**  Typically, much like these above, combinations of many features would be contrasted to realise insights from the data.  However, as marking criteria for this project draws focus to the target feature (`attrition`) only, this typical analysis is deferred as the effort would, in effect, realise no value. 

Ideas for further / typical bivariate analysis:

- [ ] Average Salary Over Time
- [ ] salary vs business travel, box plot
- [ ] salary vs education
- [ ] salary vs job role
- [ ] count emp by dept, hue=education level
- [ ] count emp by role, hue=education level
- [ ] violin plots for age, education, state

#### Analysis of target feature: `attrition`

In [ ]:
_ = dfEmployee.copy()
_.attrition = _.attrition.apply(lambda v: 'No' if v == 0 else 'Yes')

sns.histplot(data=_, y='distance_from_home_km', hue='attrition', multiple='stack')

Observations:

- Whereas one may expect those that travel the farthest may be enticed to change employer to travel less, is not borne out in analysis of the data.  This indicates either that staff are traveling not to work but for work -- see the above inferred professional or field services -- or that incentives are sufficiently set to offset such expected behaviour 

In [ ]:
sns.barplot(data=_, x='years_at_company', y='salary', hue='attrition')


Observations:

- Plainly visible from this analysis, through each year of tenure: those paid less are more likely to churn
- However, peculiarly, the variance in the lesser-paid is higher. This indicates that pay is both a driver of churn, but subjectively set creating an unreliable status for both staff and employer
- Pay trends downwards from Y1 to Y7, that is, founder and new staff are better rewarded than loyal staff

In [ ]:
barh_specs = [
    ('stock_option_level',              'attrition', False, 'Attrition Rate by Stock Option Level'),
    ('job_role',                        'attrition', False, 'Attrition Rate by Job Role'),
    ('department',                      'attrition', False, 'Attrition Rate by Department'),
    ('over_time',                       'attrition', True,  'Attrition Rate by Overtime'),
    ('business_travel',                 'attrition', False, 'Attrition Rate by Business Travel'),
    ('gender',                          'attrition', False, 'Attrition Rate by Gender'),
    ('ethnicity',                       'attrition', False, 'Attrition Rate by Ethnicity'),
    ('state',                           'attrition', False, 'Attrition Rate by State'),
    ('education_level',                 'attrition', False, 'Attrition Rate by Education'),
    ('education_field',                 'attrition', False, 'Attrition Rate by Education Field'),
    
    ('environment_satisfaction_level',  'attrition', False, 'Attrition Rate by Environment Satisfaction'),
    ('job_satisfaction_level',          'attrition', False, 'Attrition Rate by Job Satisfaction'),
    ('relationship_satisfaction_level', 'attrition', False, 'Attrition Rate by Relationship Satisfaction'),
    ('work_life_balance_level',         'attrition', False, 'Attrition Rate by Work-Life Balance Satisfaction'),
    ('self_rating_level',               'attrition', False, 'Attrition Rate by Self Rating'),
    ('manager_rating_level',            'attrition', False, 'Attrition Rate by Manager Rating'),
]


plt.figure(figsize=(20, 20))
for i, (group, target, both, title) in enumerate(barh_specs, 1):
    dfEmployee.groupby(group, observed=True)[target].value_counts(normalize=True).unstack() \
        .rename(columns={0: 'No', 1: 'Yes'}, index={0: 'No', 1: 'Yes'} if both else None) \
        .apply(lambda v: v * 100) \
        .plot(kind='barh', stacked=True, title=title, ax=plt.subplot(5, 4, i), legend=False)

plt.legend()
plt.tight_layout()

Observations:

- Particular roles -- sales representative, recruiter, and data scientist -- are particularly prone to churn relative to their peers
- Overtime (work condition) is, as expected, more likely to foster staff churn
- Minority ethnicities are more likely to foster staff churn
- As expected, `3 Neutral` or `1 Very Dissatisfied` survey answers is correlated more that others with staff churn

### Correlation Analysis

In [ ]:
sns.heatmap(data=dfEmployee.corr(numeric_only=True), cmap='coolwarm')

Observation:

- While `years_*` (scalar) features are correlating with each other, as one may expect, none are correlating with `attrition`
- Categorical features, therefore, are expected to be significant for predicting `attrition`, as borne out by the modelling below

## Modelling

In [55]:
seed = 42
target_feature = 'attrition'

# tuning parameters
folds = 5
C_range = [0.00001, 0.0001, 0.001, 0.01, 0.1, 1.0]  # a list comprehension isn't as readable
estimators_range = range(10, 101, 10)
max_depth_range = [2, 3, 5, 8, 13]
boost_eta_range = [0.3, 0.2, 0.1]

### Drop superfluous data frames & features (-> performace)

In [56]:
del dfEducation
del dfRating
del dfSatisfied

del dfPerformance
del dfPerformance_recent

In [ ]:
# del dfEmployee['gender']      # IDEA  Remove if insignificant
# del dfEmployee['ethnicity']

del dfEmployee['first_name']
del dfEmployee['last_name']
del dfEmployee['education']
del dfEmployee['hire_date']
del dfEmployee['hire_date_ym']
del dfEmployee['hire_date_yq']
del dfEmployee['performance_id']
del dfEmployee['review_date']
del dfEmployee['review_date_ym']
del dfEmployee['review_date_yq']

del dfEmployee['environment_satisfaction']
del dfEmployee['job_satisfaction']
del dfEmployee['relationship_satisfaction']
del dfEmployee['work_life_balance']

del dfEmployee['self_rating']
del dfEmployee['manager_rating']

display(dfEmployee.head().T)

display('scalar_features:', dfEmployee[scalar_features(dfEmployee)].info())
display('categorical_features:', dfEmployee[categorical_features(dfEmployee)].info())

### Fill empty features

In [ ]:
dfEmployee.isna().sum()

In [59]:
dfEmployee.fillna({
    'training_opportunities_within_year': 0, 
    'training_opportunities_taken': 0,
    }, inplace=True)

In [ ]:
dfEmployee.isna().sum()

### Split the data

#### Training, Testing, Validation, & Full (Training + Validation)

In [ ]:
df_val, df_test, df_train, df_full = validation_testing_training_full_split(dfEmployee, seed=seed)

nTotal = len(dfEmployee)
nVal = len(df_val)
nTest = len(df_test)
nTrain = len(df_train)
nFull = len(df_full)

round(nVal/nTotal, 1), round(nTest/nTotal, 1), round(nTrain/nTotal, 1), round(nFull/nTotal,1), round(nTotal/nTotal)

#### Split out `y` (target feature) from all datasets 

In [62]:
df_val, y_val = y_split(df_val, yColumn=target_feature)
df_test, y_test = y_split(df_test, yColumn=target_feature)
df_train, y_train = y_split(df_train, yColumn=target_feature)
df_full, y_full = y_split(df_full, yColumn=target_feature)

assert df_val.shape[1] == df_test.shape[1] and df_test.shape[1] == df_train.shape[1] and df_train.shape[1] == df_full.shape[1]
assert len(y_val) == df_val.shape[0] and len(y_test) == df_test.shape[0] and len(y_train) == df_train.shape[0] and len(y_full) == df_full.shape[0]

### Logistic Regression Model (as `y_*` is cateogrical)

In [63]:
model, dv = fit(
    model=LogisticRegression(solver='liblinear', max_iter=1000, random_state=seed, C=1.0),
    df=df_train,
    y=y_train
)
y_val_pred = predict(model, dv, df_val)

In [ ]:
fpr, tpr, _ = roc_curve(y_val, y_val_pred)
roc_auc_score(y_val, y_val_pred).round(3)

Observation:

- 77% ROC AUC is quite welcomed; sets a baseline for tuning (higher is better)

In [ ]:
plt.figure(figsize=(5, 5))

plt.plot(fpr, tpr, label='model')
plt.plot([0, 1], [0, 1], label='random', linestyle='--')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.legend()

#### Linear regression model tuning

In [ ]:
scores = []

for C in C_range:
    kfold = KFold(n_splits=folds, shuffle=True, random_state=seed)
    fold = 1
    
    for training_indices, validation_indices in tqdm(kfold.split(df_full), desc=f'{C = :>6}'):
        df_training   = df_full.iloc[training_indices]
        y_training    = y_full .iloc[training_indices]
        df_validation = df_full.iloc[validation_indices]
        y_validation  = y_full . iloc[validation_indices]
        
        model, dv = fit(
            model=LogisticRegression(solver='liblinear', max_iter=1000, random_state=seed, C=C),
            df=df_training,
            y=y_training
        )        
        y_pred = predict(model, dv, df_validation)
        
        scores.append((C, fold, roc_auc_score(y_validation, y_pred)))
        fold += 1
        
_ = pd.DataFrame(scores, columns=['C', 'fold', 'auc'])
display(_.groupby(by='C').max())


best_C = _.groupby(by='C').max().sort_values(by='auc', ascending=False).index[0]
best_C

In [ ]:
model, dv = fit(
    model=LogisticRegression(solver='liblinear', max_iter=1000, random_state=seed, C=best_C),
    df=df_training,
    y=y_training
)

pd.DataFrame(zip(dv.feature_names_, abs(model.coef_[0])), columns=['feature', 'importance']) \
    .sort_values(by='importance', ascending=False) \
        .tail(20)

Observation:

- `salary`, `ethnicity`, and `job_role` least significant features 

In [ ]:
scores = []
dropped_features = ['salary', 'gender', 'ethnicity', 'job_role']

# for dropped_feature_combo in [list(combo) for combo in [[], *combinations(dropped_features, 1), *combinations(dropped_features, 2), *combinations(dropped_features, 3), dropped_features]]:
for dropped_feature_combo in all_combinations_of(dropped_features):
    # display(dropped_feature_combo)
    for C in C_range:
        kfold = KFold(n_splits=folds, shuffle=True, random_state=seed)
        fold = 1
        
        for training_indices, validation_indices in tqdm(kfold.split(df_full), desc=f'{C = :>6}, {dropped_feature_combo = }'):
            df_training   = df_full.iloc[training_indices]
            y_training    = y_full .iloc[training_indices]
            df_validation = df_full.iloc[validation_indices]
            y_validation  = y_full . iloc[validation_indices]
            
            model, dv = fit(
                model=LogisticRegression(solver='liblinear', max_iter=1000, random_state=seed, C=C),
                df=df_training,
                y=y_training,
                drop=dropped_feature_combo
            )        
            y_pred = predict(model, dv, df_validation, drop=dropped_feature_combo)
            
            scores.append((dropped_feature_combo, C, fold, roc_auc_score(y_validation, y_pred)))
            fold += 1
        
_ = pd.DataFrame(scores, columns=['dropped_features', 'C', 'fold', 'auc']) \
    .groupby(by='C').max().sort_values(by='auc', ascending=False)
display(_)

best_C = _.index[0]
dropped_features = _.iloc[0]['dropped_features']

best_tuning = {
    'logistic': {
        'seed': seed,
        'dropped_features': dropped_features,
        'C': best_C,
        }
    }
dropped_features, best_C

In [ ]:
model, dv = fit(
    model=LogisticRegression(solver='liblinear', max_iter=1000, random_state=seed, C=best_C),
    df=df_train,
    y=y_train,
    drop=dropped_features
)

y_val_pred = predict(model, dv, df_val, drop=dropped_features)
fpr, tpr, _ = roc_curve(y_val, y_val_pred)
roc_auc_score(y_val, y_val_pred).round(3)

Observation:

- 97% ROC AUC is very good (perhaps too good?), achieved by dropping `dropped_features` and setting C=`best_C`

In [ ]:
plt.figure(figsize=(5, 5))

plt.plot(fpr, tpr, label='model')
plt.plot([0, 1], [0, 1], label='random', linestyle='--')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.legend()

### Random Forest Evaluation Model

In [ ]:
model, dv = fit(
    model=RandomForestRegressor(n_estimators=10, random_state=seed, n_jobs=-1), 
    dv=DictVectorizer(sparse=True),
    df=df_train, 
    y=y_train,
)
y_val_pred = decide(model, dv, df_val)
root_mean_squared_error(y_val, y_val_pred).round(3)

#### Random Forest model tuning

In [ ]:
pd.DataFrame(zip(dv.feature_names_, model.feature_importances_), columns=['feature', 'importance']) \
    .sort_values(by='importance', ascending=False) \
        .tail(20)

In [ ]:
scores = []
dropped_features = ['salary', 'gender', 'ethnicity', 'job_role'] # for consistency with Logistic Regression

for dropped_feature_combo in all_combinations_of(dropped_features):
    # display(dropped_feature_combo)
    for estimators in estimators_range:
        for max_depth in tqdm(max_depth_range, f'{dropped_feature_combo = } {estimators= :>2}'):
            model, dv = fit(
                model=RandomForestRegressor(max_depth=max_depth, n_estimators=estimators, random_state=seed, n_jobs=-1), 
                dv=DictVectorizer(sparse=True),
                df=df_train, 
                y=y_train,
                drop=dropped_feature_combo
            )
            
            y_val_pred = decide(model, dv, df_val, drop=dropped_feature_combo)
            
            scores.append(('-'.join(dropped_feature_combo), max_depth, estimators, root_mean_squared_error(y_val, y_val_pred)))
        
_ = pd.DataFrame(scores, columns=['dropped_features', 'max_depth', 'estimators', 'rmse']) \
    .sort_values(by='rmse')
display(_.head(10))

dropped_features, best_depth, best_estimators = _.iloc[0, 0:3]  # First row, first 3 columns
dropped_features = dropped_features.split('-')
best_tuning['random_forest'] = {
    'seed': seed,
    'dropped_features': dropped_features,
    'depth': best_depth,
    'estimators': best_estimators,
    }
dropped_features, best_depth, best_estimators

In [ ]:
model, dv = fit(
    model=RandomForestRegressor(max_depth=best_depth, n_estimators=best_estimators, random_state=seed, n_jobs=-1), 
    dv=DictVectorizer(sparse=True),
    df=df_train, 
    y=y_train,
    drop=dropped_features
)
            
y_val_pred = decide(model, dv, df_val, drop=dropped_features)
fpr, tpr, _ = roc_curve(y_val, y_val_pred)
roc_auc_score(y_val, y_val_pred).round(3)

Observation:

- 96% ROC AUC is very good (and consistent with the tuned Logistic Regression model), achieved by dropping dropped_features=`dropped_features` and setting max_depth=`best_depth` and n_estimators=`best_estimators`

In [ ]:
plt.figure(figsize=(5, 5))

plt.plot(fpr, tpr, label='model')
plt.plot([0, 1], [0, 1], label='random', linestyle='--')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.legend()

### Gradient Boosting model

In [ ]:
scores = []

for eta in boost_eta_range:
    for depth in tqdm(max_depth_range, desc=f'{eta = }'):
        xgb_params = {
            'eta': eta, 
            'max_depth': depth,
            'min_child_weight': 1,
            
            'objective': 'reg:squarederror',
            'nthread': 2,
            
            'seed': seed,
            'verbosity': 1,
        }
        
        X_train, dv = one_hot_encode(df_train, dv=DictVectorizer(sparse=True), fit=True)
        X_val, _ = one_hot_encode(df_val, dv=dv)
        
        dX_train = xgb.DMatrix(X_train, label=y_train, feature_names=dv.feature_names_)
        dX_val = xgb.DMatrix(X_val, label=y_val, feature_names=dv.feature_names_)
        
        model = xgb.train(xgb_params, dX_train, num_boost_round=100)
        y_val_pred = model.predict(dX_val)
        
        scores.append((eta, depth, xgb_params, root_mean_squared_error(y_val, y_val_pred)))
        
_ = pd.DataFrame(scores, columns=['eta', 'max_depth', 'xgb_params', 'rmse']) \
    .sort_values(by='rmse')
display(_.head(10))

best_xgb_params = _.iloc[0]['xgb_params']
best_tuning['boost'] = best_xgb_params
best_xgb_params

In [ ]:
X_train, dv = one_hot_encode(df_train, dv=DictVectorizer(sparse=True), fit=True)
X_val, _ = one_hot_encode(df_val, dv=dv)

dX_train = xgb.DMatrix(X_train, label=y_train, feature_names=dv.feature_names_)
dX_val = xgb.DMatrix(X_val, label=y_val, feature_names=dv.feature_names_)

model = xgb.train(best_xgb_params, dX_train, num_boost_round=100)
y_val_pred = model.predict(dX_val)

fpr, tpr, _ = roc_curve(y_val, y_val_pred)
roc_auc_score(y_val, y_val_pred).round(3)

Observation:

- 97% ROC AUC is very good (and consistent with the tuned Logistic Regression and Decision Tree models), achieved by setting xgb_params=`best_xgb_params`

In [ ]:
plt.figure(figsize=(5, 5))

plt.plot(fpr, tpr, label='model')
plt.plot([0, 1], [0, 1], label='random', linestyle='--')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.legend()

### Model Selection

When comparing the plotted False Positive vs False Negative rates, the **Logistic Regression** model is optimal despite it's ROC AUC score being the lowest of an exceptionally good bunch.

In [ ]:
display(best_tuning)

dropped_features = best_tuning['logistic']['dropped_features']
best_C = best_tuning['logistic']['C']
seed = best_tuning['logistic']['seed']

dropped_features, best_C, seed

In [ ]:
model, dv = fit(
    model=LogisticRegression(solver='liblinear', max_iter=1000, random_state=seed, C=best_C),
    df=df_full,
    y=y_full,
    drop=dropped_features
)

model, dv

**NOTE**:  Please refer to `./train_and_persist.py` for continued model persistence into `./models/` 